# 04 Data Merging

Notebook ini merge 3 dataset clean dari `data/clean`:
- `daily_household_transactions_clean.csv`
- `ecommerce_sales_clean.csv`
- `personal_finance_clean.csv`

Tujuan:
- Baca 3 dataset clean.
- Samakan nama fitur utama.
- Gabungkan jadi 1 dataset utuh.
- Cek hasil merge.
- Buat tabel fitur: mana dipertahankan, mana kandidat buang nanti.
- Simpan output ke `data/clean/merged_finance_ecommerce_household_clean.csv`.

Catatan logic:
- Merge pakai `pd.concat`, bukan join. Sebab 3 dataset berisi transaksi berbeda, bukan record yang perlu dicocokkan pakai key.
- Fitur sumber (`sumber_dataset`) wajib dipertahankan agar asal data tetap bisa dilacak.


## 1. Import library

Pakai `pandas` untuk merge, `matplotlib` untuk cek visual singkat.


In [14]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-whitegrid')


## 2. Konfigurasi path

Input dari `data/clean`. Output juga disimpan ke `data/clean` sebagai 1 dataset gabungan.


In [15]:
root_project = Path.cwd().resolve()
if root_project.name == 'notebooks':
    root_project = root_project.parent

folder_clean = root_project / 'data' / 'clean'
file_daily = folder_clean / 'daily_household_transactions_clean.csv'
file_ecommerce = folder_clean / 'ecommerce_sales_clean.csv'
file_finance = folder_clean / 'personal_finance_clean.csv'
file_output = folder_clean / 'merged_finance_ecommerce_household_clean.csv'

for file_path in [file_daily, file_ecommerce, file_finance]:
    if not file_path.exists():
        raise FileNotFoundError(f'File clean belum ada: {file_path}')

print('Folder clean:', folder_clean)
print('Output file :', file_output)


Folder clean: /home/umaygans/05_nayyara_submission_1/nayyara_capstone/data/clean
Output file : /home/umaygans/05_nayyara_submission_1/nayyara_capstone/data/clean/merged_finance_ecommerce_household_clean.csv


## 3. Load 3 dataset clean

Cell ini baca semua dataset dan tampilkan shape + kolom. Ini penting sebelum mapping fitur.


In [16]:
data_daily = pd.read_csv(file_daily)
data_ecommerce = pd.read_csv(file_ecommerce)
data_finance = pd.read_csv(file_finance)

ringkasan_input = pd.DataFrame([
    {'sumber_dataset': 'daily_household', 'baris': len(data_daily), 'kolom': data_daily.shape[1], 'file': file_daily.name},
    {'sumber_dataset': 'ecommerce_sales', 'baris': len(data_ecommerce), 'kolom': data_ecommerce.shape[1], 'file': file_ecommerce.name},
    {'sumber_dataset': 'personal_finance', 'baris': len(data_finance), 'kolom': data_finance.shape[1], 'file': file_finance.name},
])

display(ringkasan_input)

for nama, data in [('daily_household', data_daily), ('ecommerce_sales', data_ecommerce), ('personal_finance', data_finance)]:
    print('')
    print(nama)
    print(list(data.columns))


,sumber_dataset,baris,kolom,file
0,daily_household,2452,19,daily_household_transactions_clean.csv
1,ecommerce_sales,4545,28,ecommerce_sales_clean.csv
2,personal_finance,1500,7,personal_finance_clean.csv



daily_household
['transaction_date', 'year', 'month', 'year_month', 'transaction_type', 'direction', 'payment_mode', 'currency', 'category_raw', 'category_clean', 'subcategory_raw', 'note', 'amount_inr', 'amount_idr', 'amount_signed_inr', 'amount_signed_idr', 'is_outlier_amount', 'amount_inr_capped', 'amount_idr_capped']

ecommerce_sales
['order_id', 'dataset_id', 'nama_dataset', 'periode', 'tanggal_pesanan', 'tanggal_diisi_dari_median', 'status_pesanan', 'alasan_pembatalan', 'kategori_produk', 'jumlah_kategori_produk', 'jumlah_barang', 'berat_total_gram', 'jumlah_return', 'diskon_total', 'metode_pembayaran', 'opsi_pengiriman', 'kota_kabupaten', 'provinsi', 'ongkir_dibayar_pembeli', 'potongan_ongkir', 'total_pembayaran', 'perkiraan_ongkir', 'tahun', 'bulan', 'tahun_bulan', 'kategori_umum', 'outlier_total_pembayaran', 'total_pembayaran_capped']

personal_finance
['tanggal_transaksi', 'tahun', 'bulan', 'tahun_bulan', 'tipe_transaksi', 'kategori_clean', 'amount_idr']


## 4. Fitur final gabungan

Logic fitur:
- Fitur inti dipakai semua dataset: tanggal, tahun, bulan, kategori, amount, outlier, sumber.
- Fitur detail dipertahankan walau hanya ada di sebagian dataset: order id, payment method, lokasi, quantity, note.
- Fitur yang tidak relevan pada dataset tertentu diisi kosong (`pd.NA`).


In [17]:
kolom_final = [
    'sumber_dataset', 'id_transaksi',
    'tanggal_transaksi', 'tahun', 'bulan', 'tahun_bulan',
    'tipe_transaksi',
    'kategori_clean',
    'amount_idr',
    'metode_pembayaran',
    'status_transaksi', 'sumber_file',
]

## 5. Mapping daily household

Logic mapping:
- `transaction_date` jadi `tanggal_transaksi`.
- `amount_idr`, `amount_signed_idr`, dan `amount_idr_capped` sudah ada.
- `payment_mode` jadi `metode_pembayaran`.
- Tidak punya lokasi/order id, jadi kosong.


In [18]:
def mapping_daily(data):
    hasil = pd.DataFrame(index=data.index)

    hasil['sumber_dataset'] = 'daily_household'
    hasil['id_transaksi'] = pd.NA
    hasil['tanggal_transaksi'] = pd.to_datetime(data['transaction_date'], errors='coerce')
    hasil['tahun'] = data['year']
    hasil['bulan'] = data['month']
    hasil['tahun_bulan'] = data['year_month']
    hasil['tipe_transaksi'] = data['transaction_type']
    hasil['kategori_clean'] = data['category_clean']
    hasil['amount_idr'] = data['amount_idr']
    hasil['metode_pembayaran'] = data['payment_mode']
    hasil['status_transaksi'] = data['transaction_type']
    hasil['sumber_file'] = file_daily.name

    return hasil[kolom_final]

## 6. Mapping e-commerce

Logic mapping:
- E-commerce dianggap transaksi `Sales`, arah positif (`1`).
- `total_pembayaran` jadi `amount_idr`.
- `total_pembayaran_capped` jadi `amount_idr_capped`.
- Detail e-commerce seperti `order_id`, lokasi, metode pembayaran, dan jumlah barang dipertahankan.


In [19]:
def mapping_ecommerce(data):
    hasil = pd.DataFrame(index=data.index)

    hasil['sumber_dataset'] = 'ecommerce_sales'
    hasil['id_transaksi'] = data['order_id']
    hasil['tanggal_transaksi'] = pd.to_datetime(data['tanggal_pesanan'], errors='coerce')
    hasil['tahun'] = data['tahun']
    hasil['bulan'] = data['bulan']
    hasil['tahun_bulan'] = data['tahun_bulan']
    hasil['tipe_transaksi'] = 'Sales'
    hasil['kategori_clean'] = data['kategori_umum']
    hasil['amount_idr'] = data['total_pembayaran']
    hasil['metode_pembayaran'] = data['metode_pembayaran']
    hasil['status_transaksi'] = data['status_pesanan']
    hasil['sumber_file'] = file_ecommerce.name

    return hasil[kolom_final]

## 7. Mapping personal finance

Logic mapping:
- Pakai schema personal finance terbaru: `tanggal_transaksi`, `tahun`, `bulan`, `tahun_bulan`, `tipe_transaksi`, `kategori_clean`, `amount_idr`.
- Fitur yang sudah dihapus dari finance (`arah`, `kategori_raw`, `deskripsi`, `outlier`, `capped`) dibuat aman di merge: derive atau isi kosong.
- Income diberi arah `1`, Expense diberi arah `-1`, supaya `amount_signed_idr` tetap bisa dipakai di dataset gabungan.


In [20]:
def mapping_finance(data):
    hasil = pd.DataFrame(index=data.index)

    hasil['sumber_dataset'] = 'personal_finance'
    hasil['id_transaksi'] = pd.NA
    hasil['tanggal_transaksi'] = pd.to_datetime(data['tanggal_transaksi'], errors='coerce')
    hasil['tahun'] = data['tahun']
    hasil['bulan'] = data['bulan']
    hasil['tahun_bulan'] = data['tahun_bulan']
    hasil['tipe_transaksi'] = data['tipe_transaksi']
    hasil['kategori_clean'] = data['kategori_clean']
    hasil['amount_idr'] = data['amount_idr']
    hasil['metode_pembayaran'] = pd.NA
    hasil['status_transaksi'] = data['tipe_transaksi']
    hasil['sumber_file'] = file_finance.name

    return hasil[kolom_final]

## 8. Merge 3 dataset

Merge vertikal: semua transaksi ditumpuk jadi satu tabel. Setelah merge, urutkan berdasarkan tanggal.


In [21]:
data_daily_merge = mapping_daily(data_daily)
data_ecommerce_merge = mapping_ecommerce(data_ecommerce)
data_finance_merge = mapping_finance(data_finance)

data_gabungan = pd.concat(
    [data_daily_merge, data_ecommerce_merge, data_finance_merge],
    ignore_index=True
)

data_gabungan = data_gabungan[kolom_final]
data_gabungan = data_gabungan.sort_values('tanggal_transaksi').reset_index(drop=True)

print('Shape data gabungan:', data_gabungan.shape)

display(data_gabungan.head())
display(data_gabungan.groupby('sumber_dataset').size().reset_index(name='jumlah_baris'))

Shape data gabungan: (8497, 12)


,sumber_dataset,id_transaksi,tanggal_transaksi,tahun,bulan,tahun_bulan,tipe_transaksi,kategori_clean,amount_idr,metode_pembayaran,status_transaksi,sumber_file
0,daily_household,NaN,2015-01-01,2015,1,2015-01,Expense,Transportasi,1830.0,Cash,Expense,daily_household_transactions_clean.csv
1,daily_household,NaN,2015-01-01,2015,1,2015-01,Expense,Makanan,73200.0,Credit Card,Expense,daily_household_transactions_clean.csv
2,daily_household,NaN,2015-01-01,2015,1,2015-01,Expense,Transportasi,3660.0,Cash,Expense,daily_household_transactions_clean.csv
3,daily_household,NaN,2015-01-01,2015,1,2015-01,Expense,Transportasi,10980.0,Cash,Expense,daily_household_transactions_clean.csv
4,daily_household,NaN,2015-01-01,2015,1,2015-01,Expense,Pendidikan & Dokumen,7320.0,Cash,Expense,daily_household_transactions_clean.csv


,sumber_dataset,jumlah_baris
0,daily_household,2452
1,ecommerce_sales,4545
2,personal_finance,1500


## 9. Cek hasil merge

Cek basic:
- missing kolom penting
- amount tidak valid
- distribusi sumber data
- tipe transaksi


In [22]:
kolom_wajib = [
    'sumber_dataset',
    'tanggal_transaksi',
    'tahun_bulan',
    'tipe_transaksi',
    'kategori_clean',
    'amount_idr'
]

cek_missing = pd.DataFrame({
    'kolom': kolom_wajib,
    'missing': [int(data_gabungan[kolom].isna().sum()) for kolom in kolom_wajib]
})

cek_missing['missing_pct'] = (
    cek_missing['missing'] / len(data_gabungan) * 100
).round(2)

cek_ringkas = pd.DataFrame([
    {'metric': 'jumlah_baris', 'value': len(data_gabungan)},
    {'metric': 'jumlah_kolom', 'value': data_gabungan.shape[1]},
    {'metric': 'amount_kosong', 'value': int(data_gabungan['amount_idr'].isna().sum())},
    {'metric': 'amount_tidak_valid', 'value': int((data_gabungan['amount_idr'] <= 0).sum())}
])

display(cek_missing)
display(cek_ringkas)
display(data_gabungan['tipe_transaksi'].value_counts(dropna=False).reset_index(name='jumlah_baris'))

,kolom,missing,missing_pct
0,sumber_dataset,0,0.0
1,tanggal_transaksi,0,0.0
2,tahun_bulan,0,0.0
3,tipe_transaksi,0,0.0
4,kategori_clean,0,0.0
5,amount_idr,0,0.0


,metric,value
0,jumlah_baris,8497
1,jumlah_kolom,12
2,amount_kosong,0
3,amount_tidak_valid,0


,tipe_transaksi,jumlah_baris
0,Sales,4545
1,Expense,3395
2,Income,403
3,Transfer-Out,154


## 12. Simpan data gabungan

Simpan satu dataset gabungan. Belum drop fitur kandidat, karena logic fitur masih mau dicek dulu.


In [23]:
data_gabungan.to_csv(file_output, index=False)

print('File gabungan tersimpan:', file_output)
print('Jumlah baris:', len(data_gabungan))
print('Jumlah kolom:', data_gabungan.shape[1])


File gabungan tersimpan: /home/umaygans/05_nayyara_submission_1/nayyara_capstone/data/clean/merged_finance_ecommerce_household_clean.csv
Jumlah baris: 8497
Jumlah kolom: 12


## 13. Variabel penting

- `data_daily`, `data_ecommerce`, `data_finance`: input clean.
- `data_gabungan`: hasil merge final.
- `review_fitur`: tabel diskusi fitur dipertahankan/dibuang.
- `cek_missing`, `cek_ringkas`: validasi merge.


In [24]:
pd.read_csv("/home/umaygans/05_nayyara_submission_1/nayyara_capstone/data/clean/merged_finance_ecommerce_household_clean.csv")

,sumber_dataset,id_transaksi,tanggal_transaksi,tahun,bulan,tahun_bulan,tipe_transaksi,kategori_clean,amount_idr,metode_pembayaran,status_transaksi,sumber_file
0,daily_household,NaN,2015-01-01 00:00:00,2015,1,2015-01,Expense,Transportasi,1830.0,Cash,Expense,daily_household_transactions_clean.csv
1,daily_household,NaN,2015-01-01 00:00:00,2015,1,2015-01,Expense,Makanan,73200.0,Credit Card,Expense,daily_household_transactions_clean.csv
2,daily_household,NaN,2015-01-01 00:00:00,2015,1,2015-01,Expense,Transportasi,3660.0,Cash,Expense,daily_household_transactions_clean.csv
3,daily_household,NaN,2015-01-01 00:00:00,2015,1,2015-01,Expense,Transportasi,10980.0,Cash,Expense,daily_household_transactions_clean.csv
4,daily_household,NaN,2015-01-01 00:00:00,2015,1,2015-01,Expense,Pendidikan & Dokumen,7320.0,Cash,Expense,daily_household_transactions_clean.csv
...,...,...,...,...,...,...,...,...,...,...,...,...
8492,ecommerce_sales,ORD_0016168,2025-11-30 21:36:00,2025,11,2025-11,Sales,Belanja,15900.0,COD (Bayar di Tempat),Telah Dikirim,ecommerce_sales_clean.csv
8493,ecommerce_sales,ORD_0016169,2025-11-30 21:52:00,2025,11,2025-11,Sales,Belanja,14499.0,COD (Bayar di Tempat),Telah Dikirim,ecommerce_sales_clean.csv
8494,ecommerce_sales,ORD_0016170,2025-11-30 22:04:00,2025,11,2025-11,Sales,Belanja,18000.0,COD (Bayar di Tempat),Telah Dikirim,ecommerce_sales_clean.csv
8495,ecommerce_sales,ORD_0016171,2025-11-30 22:19:00,2025,11,2025-11,Sales,Belanja,63000.0,COD (Bayar di Tempat),Sedang Dikirim,ecommerce_sales_clean.csv
